# Cálculo Paso a Paso del Índice HHI (Arquitectura Medallion)

Este notebook demuestra cómo calcular correctamente el Índice Herfindahl-Hirschman (HHI) utilizando los datos limpios de la capa **Silver** (arquitectura actual), en lugar del dataset legacy (`cruce_secop_dane.parquet`) que presentaba pérdida de datos.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.options.display.float_format = '{:,.2f}'.format

## 1. Cargar Datos Transaccionales (Silver)

El HHI requiere saber cuánto vendió **cada contratista individual** en un municipio. Por lo tanto, no podemos usar las tablas agregadas (`fact_contratacion`); debemos usar las tablas **transaccionales** de SECOP I y II.

In [ ]:

# Rutas a los datos limpios de Silver
path_secop_i = '../data/silver/silver_secop_i_transaccional.parquet'
path_secop_ii = '../data/silver/silver_secop_ii_transaccional.parquet'

print("Cargando SECOP I...")
df_i = pd.read_parquet(path_secop_i)
print(f"Registros SECOP I: {len(df_i):,}")

print("Cargando SECOP II...")
df_ii = pd.read_parquet(path_secop_ii)
print(f"Registros SECOP II: {len(df_ii):,}")

# Unir ambas bases y deduplicar igual que Gold para no inflar montos ni cuotas
df_completo = pd.concat([df_i, df_ii], ignore_index=True)
df_completo['id_contrato'] = df_completo['id_contrato'].astype(str).str.strip()
duplicados = int(df_completo.duplicated(subset=['id_contrato']).sum())
df_completo = df_completo.drop_duplicates(subset=['id_contrato'], keep='first')

print(f"\nTotal registros combinados: {len(df_i) + len(df_ii):,}")
print(f"Contratos duplicados removidos por id_contrato: {duplicados:,}")
print(f"Total contratos ?nicos: {len(df_completo):,}")


## 2. Limpieza y Filtros Esenciales

Para calcular cuotas de mercado matemáticas, descartamos:
1. Contratos sin valor o valor <= 0
2. Contratos sin proveedor identificado (`nit_contratista`)
3. Contratos sin ubicación clara (`divipola_key`)
4. Años antes de 2018 (el análisis es 2018 en adelante)

In [ ]:

# Filtros estrictos para HHI
# Se usa la misma ventana analitica documentada para el reporte interanual.
df_completo['anio_key'] = pd.to_numeric(df_completo['anio_key'], errors='coerce')
df_completo['valor_del_contrato'] = pd.to_numeric(df_completo['valor_del_contrato'], errors='coerce')
df_completo['nit_contratista'] = (
    df_completo['nit_contratista']
    .astype(str)
    .str.replace(r'\D', '', regex=True)
    .str.strip()
)

df_valido = df_completo[
    (df_completo['valor_del_contrato'] > 0) &
    (df_completo['nit_contratista'] != '') &
    (df_completo['divipola_key'].notna()) &
    (df_completo['anio_key'] >= 2018) &
    (df_completo['anio_key'] <= 2026)
].copy()
df_valido['anio_key'] = df_valido['anio_key'].astype(int)

print(f"Registros v?lidos para el HHI: {len(df_valido):,}")
print("Registros v?lidos por a?o:")
print(df_valido['anio_key'].value_counts().sort_index().to_string())


## 3. Paso A: Calcular la Cuota de Mercado de cada Contratista

Primero sumamos toda la inversión por **municipio y año** para saber el 100% del mercado. Luego sumamos la inversión por **contratista, municipio y año**.

In [ ]:
# 1. Inversión total por municipio y año (El mercado total)
mercado_total = df_valido.groupby(['anio_key', 'divipola_key'], as_index=False).agg(
    inversion_total=('valor_del_contrato', 'sum'),
    total_contratos=('id_contrato', 'count'),
    total_proveedores=('nit_contratista', 'nunique')
)

# 2. Inversión por cada contratista en ese municipio y año
ventas_proveedor = df_valido.groupby(['anio_key', 'divipola_key', 'nit_contratista'], as_index=False).agg(
    suma_proveedor=('valor_del_contrato', 'sum')
)

# Unimos para calcular el porcentaje
hhi_calc = pd.merge(ventas_proveedor, mercado_total, on=['anio_key', 'divipola_key'])

# Cuota de mercado al cuadrado (%)
hhi_calc['participacion_sq'] = ((hhi_calc['suma_proveedor'] / hhi_calc['inversion_total']) * 100) ** 2

hhi_calc.head()

## 4. Paso B: Calcular el HHI a nivel de Municipio

Sumamos todos los cuadrados de las participaciones de los proveedores dentro del mismo municipio.

In [ ]:
hhi_municipio = hhi_calc.groupby(['anio_key', 'divipola_key'], as_index=False).agg(
    HHI=('participacion_sq', 'sum'),
    inversion_total=('inversion_total', 'first'),
    total_contratos=('total_contratos', 'first'),
    total_proveedores=('total_proveedores', 'first')
)

hhi_municipio.head()

## 5. Paso C: Evolución del HHI Promedio Nacional (Para la gráfica)

Finalmente, tomamos el HHI de cada municipio y lo promediamos por año.

In [ ]:
# Para la tabla nacional, calculamos también los proveedores únicos totales por año directamente
proveedores_nacionales = df_valido.groupby('anio_key', as_index=False).agg(
    Total_Proveedores=('nit_contratista', 'nunique')
)

hhi_nacional = hhi_municipio.groupby('anio_key', as_index=False).agg(
    HHI_Promedio=('HHI', 'mean'),
    Total_Municipios=('divipola_key', 'count'),
    Total_Contratos=('total_contratos', 'sum'),
    Suma_Inversion=('inversion_total', 'sum')
)

hhi_nacional = pd.merge(hhi_nacional, proveedores_nacionales, on='anio_key')

print("--- RESULTADOS FINALES CORRECTOS NACIONALES ---")
hhi_nacional

## 6. Evolución del HHI Solo en Bogotá (11001)

Filtramos la tabla de municipios para quedarnos únicamente con la evolución del HHI en Bogotá a lo largo de los años.

In [ ]:
# El código DIVIPOLA de Bogotá D.C. es '11001'
hhi_bogota = hhi_municipio[hhi_municipio['divipola_key'] == '11001'].copy()

hhi_bogota = hhi_bogota.sort_values('anio_key')

print("--- HHI BOGOTÁ D.C. (11001) ---")
hhi_bogota[['anio_key', 'HHI', 'total_proveedores', 'total_contratos', 'inversion_total']]


## 7. Ejercicio Adicional: Filtro de Atípicos y Segmentación por Tipo de Entidad

Como la capa Silver transaccional está simplificada, para segmentar por nivel (Nacional vs Territorial) necesitamos hacer un *join* con la capa **Bronze** para rescatar la columna `orden_entidad`. 

Luego, aplicaremos un filtro estadístico para quitar contratos atípicos (outliers) utilizando el método del Rango Intercuartílico (IQR) o removiendo el 1% superior.

In [ ]:

# Extraer Orden Entidad desde Bronze como fallback para Silver historico.
# La ingesta corregida ya debe escribir orden_entidad en Silver; este bloque conserva
# reproducibilidad mientras existan parquets Silver generados antes de esa correccion.
path_bronze_i = '../data/bronze/secop_i/secop_i_raw.parquet'
path_bronze_ii = '../data/bronze/secop_ii/secop_ii_raw.parquet'

def clasificar_orden(valor):
    raw = '' if pd.isna(valor) else str(valor).upper().strip()
    if 'NACIONAL' in raw:
        return 'NACIONAL'
    if 'TERRITORIAL' in raw or 'DISTRITO CAPITAL' in raw or 'AREA METROPOLITANA' in raw:
        return 'TERRITORIAL'
    if 'CORPOR' in raw or 'AUTONOMA' in raw or 'AUT?NOMA' in raw:
        return 'OTRO'
    if raw == '' or raw == 'NAN' or 'NO DEFINIDO' in raw:
        return 'NO_DEFINIDO'
    return 'OTRO'

frames_orden = []
try:
    bronze_i = pd.read_parquet(path_bronze_i, columns=['UID', 'Orden Entidad'])
    bronze_i = bronze_i.rename(columns={'UID': 'id_contrato', 'Orden Entidad': 'orden_raw'})
    frames_orden.append(bronze_i)
except Exception as exc:
    print(f"No se pudo leer Orden Entidad de SECOP I Bronze: {exc}")

try:
    # Nombre real en Bronze II: 'ID Contrato' y 'Orden'.
    bronze_ii = pd.read_parquet(path_bronze_ii, columns=['ID Contrato', 'Orden'])
    bronze_ii = bronze_ii.rename(columns={'ID Contrato': 'id_contrato', 'Orden': 'orden_raw'})
    frames_orden.append(bronze_ii)
except Exception as exc:
    print(f"No se pudo leer Orden de SECOP II Bronze: {exc}")

if frames_orden:
    bronze_entidad = pd.concat(frames_orden, ignore_index=True)
    bronze_entidad['id_contrato'] = bronze_entidad['id_contrato'].astype(str).str.strip()
    bronze_entidad['orden_entidad'] = bronze_entidad['orden_raw'].map(clasificar_orden)
    bronze_entidad = bronze_entidad[['id_contrato', 'orden_entidad']].drop_duplicates('id_contrato')
else:
    bronze_entidad = pd.DataFrame(columns=['id_contrato', 'orden_entidad'])

if 'orden_entidad' in df_valido.columns:
    df_valido['orden_entidad'] = df_valido['orden_entidad'].map(clasificar_orden)
else:
    df_valido['orden_entidad'] = 'NO_DEFINIDO'

df_entidad = pd.merge(df_valido, bronze_entidad, on='id_contrato', how='left', suffixes=('', '_bronze'))
mask = df_entidad['orden_entidad'].eq('NO_DEFINIDO') & df_entidad['orden_entidad_bronze'].notna()
df_entidad.loc[mask, 'orden_entidad'] = df_entidad.loc[mask, 'orden_entidad_bronze']
df_entidad = df_entidad.drop(columns=['orden_entidad_bronze'])
df_entidad['orden_entidad'] = df_entidad['orden_entidad'].fillna('NO_DEFINIDO')

print(f"Registros tras asignar orden_entidad: {len(df_entidad):,}")
print(df_entidad['orden_entidad'].value_counts(dropna=False).to_string())


### Filtrar Valores Atípicos (Outliers Extremos)
Para evitar que megaproyectos (contratos billonarios) distorsionen el HHI, quitaremos el 1% de los contratos más caros por cada año.

In [ ]:
# Calcular el umbral del percentil 99 por año
umbrales = df_entidad.groupby('anio_key')['valor_del_contrato'].quantile(0.99).reset_index()
umbrales.rename(columns={'valor_del_contrato': 'umbral_99'}, inplace=True)

df_sin_atipicos = pd.merge(df_entidad, umbrales, on='anio_key')
df_sin_atipicos = df_sin_atipicos[df_sin_atipicos['valor_del_contrato'] <= df_sin_atipicos['umbral_99']]

print(f"Registros originales: {len(df_entidad):,}")
print(f"Registros sin el 1% atípico: {len(df_sin_atipicos):,}")

### Recalcular el HHI segmentado y limpio

In [ ]:
# 1. Mercado Total (Segmentado por Orden de Entidad)
mercado_seg = df_sin_atipicos.groupby(['anio_key', 'divipola_key', 'orden_entidad'], as_index=False).agg(
    inversion_total=('valor_del_contrato', 'sum'),
    total_contratos=('id_contrato', 'count'),
    total_proveedores=('nit_contratista', 'nunique')
)

# 2. Ventas Proveedor
ventas_prov_seg = df_sin_atipicos.groupby(['anio_key', 'divipola_key', 'orden_entidad', 'nit_contratista'], as_index=False).agg(
    suma_proveedor=('valor_del_contrato', 'sum')
)

# Participación
hhi_calc_seg = pd.merge(ventas_prov_seg, mercado_seg, on=['anio_key', 'divipola_key', 'orden_entidad'])
hhi_calc_seg['participacion_sq'] = ((hhi_calc_seg['suma_proveedor'] / hhi_calc_seg['inversion_total']) * 100) ** 2

# HHI Municipal Segmentado
hhi_muni_seg = hhi_calc_seg.groupby(['anio_key', 'divipola_key', 'orden_entidad'], as_index=False).agg(
    HHI=('participacion_sq', 'sum'),
    inversion_total=('inversion_total', 'first'),
    total_contratos=('total_contratos', 'first'),
    total_proveedores=('total_proveedores', 'first')
)

# HHI Nacional Segmentado por Entidad
proveedores_nac_seg = df_sin_atipicos.groupby(['anio_key', 'orden_entidad'], as_index=False).agg(
    Total_Proveedores=('nit_contratista', 'nunique')
)

hhi_nac_seg = hhi_muni_seg.groupby(['anio_key', 'orden_entidad'], as_index=False).agg(
    HHI_Promedio=('HHI', 'mean'),
    Total_Municipios=('divipola_key', 'count'),
    Total_Contratos=('total_contratos', 'sum'),
    Suma_Inversion=('inversion_total', 'sum')
)

hhi_nac_seg = pd.merge(hhi_nac_seg, proveedores_nac_seg, on=['anio_key', 'orden_entidad'])

print("--- HHI PROMEDIO NACIONAL (SIN ATÍPICOS) POR TIPO DE ENTIDAD ---")
hhi_nac_seg.sort_values(['anio_key', 'orden_entidad'])

### Análisis 2: Por Año SIN Atípicos (Nacional General)

Calculamos el HHI consolidando todo el mercado por año, pero habiendo excluido el 1% de contratos más caros.

In [ ]:
# Mercado Total (Sin Atípicos, Solo por Año)
mercado_sa = df_sin_atipicos.groupby(['anio_key', 'divipola_key'], as_index=False).agg(
    inversion_total=('valor_del_contrato', 'sum'),
    total_contratos=('id_contrato', 'count'),
    total_proveedores=('nit_contratista', 'nunique')
)

# Ventas Proveedor
ventas_prov_sa = df_sin_atipicos.groupby(['anio_key', 'divipola_key', 'nit_contratista'], as_index=False).agg(
    suma_proveedor=('valor_del_contrato', 'sum')
)

# Participación
hhi_calc_sa = pd.merge(ventas_prov_sa, mercado_sa, on=['anio_key', 'divipola_key'])
hhi_calc_sa['participacion_sq'] = ((hhi_calc_sa['suma_proveedor'] / hhi_calc_sa['inversion_total']) * 100) ** 2

# HHI Municipal
hhi_muni_sa = hhi_calc_sa.groupby(['anio_key', 'divipola_key'], as_index=False).agg(
    HHI=('participacion_sq', 'sum'),
    inversion_total=('inversion_total', 'first'),
    total_contratos=('total_contratos', 'first'),
    total_proveedores=('total_proveedores', 'first')
)

# HHI Nacional
proveedores_nac_sa = df_sin_atipicos.groupby('anio_key', as_index=False).agg(
    Total_Proveedores=('nit_contratista', 'nunique')
)

hhi_nac_sa = hhi_muni_sa.groupby('anio_key', as_index=False).agg(
    HHI_Promedio=('HHI', 'mean'),
    Total_Municipios=('divipola_key', 'count'),
    Total_Contratos=('total_contratos', 'sum'),
    Suma_Inversion=('inversion_total', 'sum')
)
hhi_nac_sa = pd.merge(hhi_nac_sa, proveedores_nac_sa, on='anio_key')

print("--- HHI PROMEDIO NACIONAL (SIN ATÍPICOS) POR AÑO ---")
hhi_nac_sa.sort_values('anio_key')

### Análisis 3: Por Tipo de Entidad y Año CON Atípicos

Calculamos el HHI segmentado por orden de entidad, pero manteniendo absolutamente todos los contratos (sin importar qué tan grandes sean).

In [ ]:
# Utilizamos 'df_entidad' que NO ha sido filtrado por el umbral del 99%

# Mercado Total (Con Atípicos, Segmentado por Entidad)
mercado_ca = df_entidad.groupby(['anio_key', 'divipola_key', 'orden_entidad'], as_index=False).agg(
    inversion_total=('valor_del_contrato', 'sum'),
    total_contratos=('id_contrato', 'count'),
    total_proveedores=('nit_contratista', 'nunique')
)

# Ventas Proveedor
ventas_prov_ca = df_entidad.groupby(['anio_key', 'divipola_key', 'orden_entidad', 'nit_contratista'], as_index=False).agg(
    suma_proveedor=('valor_del_contrato', 'sum')
)

# Participación
hhi_calc_ca = pd.merge(ventas_prov_ca, mercado_ca, on=['anio_key', 'divipola_key', 'orden_entidad'])
hhi_calc_ca['participacion_sq'] = ((hhi_calc_ca['suma_proveedor'] / hhi_calc_ca['inversion_total']) * 100) ** 2

# HHI Municipal
hhi_muni_ca = hhi_calc_ca.groupby(['anio_key', 'divipola_key', 'orden_entidad'], as_index=False).agg(
    HHI=('participacion_sq', 'sum'),
    inversion_total=('inversion_total', 'first'),
    total_contratos=('total_contratos', 'first'),
    total_proveedores=('total_proveedores', 'first')
)

# HHI Nacional
proveedores_nac_ca = df_entidad.groupby(['anio_key', 'orden_entidad'], as_index=False).agg(
    Total_Proveedores=('nit_contratista', 'nunique')
)

hhi_nac_ca = hhi_muni_ca.groupby(['anio_key', 'orden_entidad'], as_index=False).agg(
    HHI_Promedio=('HHI', 'mean'),
    Total_Municipios=('divipola_key', 'count'),
    Total_Contratos=('total_contratos', 'sum'),
    Suma_Inversion=('inversion_total', 'sum')
)
hhi_nac_ca = pd.merge(hhi_nac_ca, proveedores_nac_ca, on=['anio_key', 'orden_entidad'])

print("--- HHI PROMEDIO NACIONAL (CON ATÍPICOS) POR TIPO DE ENTIDAD ---")
hhi_nac_ca.sort_values(['anio_key', 'orden_entidad'])